# Final Model Selection for Bike-Sharing Demand Prediction

After building and refining multiple regression models, the next step is to identify the best-performing model for deployment.

Model selection is not based solely on performance metrics, but also considers:

- Model accuracy
- Generalization capability
- Stability
- Interpretability

In this notebook, we:

- Compare baseline and refined models
- Evaluate regularized models (Ridge, Lasso, Elastic Net)
- Select the best model based on performance metrics
- Prepare the final model for deployment

This stage represents the transition from model experimentation to production readiness.

## Import Required Libraries

We use:

- `dplyr`: data manipulation  
- `caret`: model evaluation  
- `glmnet`: regularization models
- `readr` : reading csv files  

In [1]:
# Load required libraries

# dplyr for data manipulation
library(dplyr)

# caret for data splitting and evaluation
library(caret)

# glmnet for regularized regression models
library(glmnet)

# readr for reading the csv files
library(readr)



Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

Loading required package: ggplot2
Loading required package: lattice
Loading required package: Matrix
Loaded glmnet 4.1-10


In [2]:
library(tidyverse)
library(lubridate)
library(glmnet)
library(broom)

# Load cleaned dataset produced by 02_data_wrangling_R.ipynb
bike_data <- read.csv("clean_bike_data.csv")
bike_data$date <- as.Date(bike_data$date)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.4     ✔ tibble    3.2.1
✔ purrr     1.0.4     ✔ tidyr     1.3.1── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::expand() masks Matrix::expand()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
✖ purrr::lift()   masks caret::lift()
✖ tidyr::pack()   masks Matrix::pack()
✖ tidyr::unpack() masks Matrix::unpack()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

## Load Dataset and Prepare Data

In [3]:
# Convert character columns to proper factors BEFORE the train/test split
# This ensures both splits see all factor levels, preventing single-level errors
bike_data <- bike_data %>%
  mutate(
    # seasons has four levels: Spring, Summer, Autumn, Winter
    seasons         = factor(seasons,
                             levels = c("Spring", "Summer", "Autumn", "Winter")),
    # holiday has two levels
    holiday         = factor(holiday,
                             levels = c("No Holiday", "Holiday")),
    # day_of_week: ensure all seven levels are present
    day_of_week     = factor(day_of_week,
                             levels = c("Monday","Tuesday","Wednesday",
                                        "Thursday","Friday","Saturday","Sunday"))
  ) %>%
  # Drop functioning_day entirely — if the dataset was filtered to functional
  # hours in notebook 02, this column has only one level ("Yes") and will crash
  # model.matrix(); it carries no information in that state
  select(-functioning_day)

# Set seed for reproducibility (same seed throughout all modelling notebooks)
set.seed(123)

# 80/20 stratified split on the target variable
train_index <- createDataPartition(bike_data$rented_bike_count,
                                   p = 0.8, list = FALSE)
train_data <- bike_data[train_index, ]
test_data  <- bike_data[-train_index, ]

## Feature Engineering for Refined Models

In [4]:
# Add polynomial and interaction features

train_data <- train_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2,
    temp_humidity = temperature * humidity
  )

test_data <- test_data %>%
  mutate(
    temperature_sq = temperature^2,
    humidity_sq = humidity^2,
    temp_humidity = temperature * humidity
  )

## Train Candidate Models

We train multiple models:

- Baseline Linear Regression
- Refined Linear Regression
- Ridge Regression
- Lasso Regression
- Elastic Net

In [5]:
# Baseline model — weather predictors only (no dot notation)
baseline_model <- lm(
  rented_bike_count ~ temperature + humidity + wind_speed,
  data = train_data
)

# Refined model — adds polynomial and interaction terms
refined_model <- lm(
  rented_bike_count ~ temperature + humidity + wind_speed +
    temperature_sq + humidity_sq + temp_humidity,
  data = train_data
)

# Prepare model matrices for glmnet — explicitly name predictors
# instead of using ~ . so we control exactly what goes in
predictor_formula <- rented_bike_count ~ temperature + humidity +
  wind_speed + visibility + solar_radiation + rainfall + snowfall +
  temperature_sq + humidity_sq + temp_humidity +
  seasons + holiday + day_of_week + hour

# model.matrix converts factors to dummy columns automatically
x_train <- model.matrix(predictor_formula, data = train_data)[, -1]
# [, -1] removes the intercept column that model.matrix adds by default
y_train <- train_data$rented_bike_count

x_test  <- model.matrix(predictor_formula, data = test_data)[, -1]

# Ridge regression — alpha = 0 means pure L2 penalty
ridge_model   <- cv.glmnet(x_train, y_train, alpha = 0)

# Lasso regression — alpha = 1 means pure L1 penalty (can zero out predictors)
lasso_model   <- cv.glmnet(x_train, y_train, alpha = 1)

# Elastic Net — alpha = 0.5 balances L1 and L2 penalties
elastic_model <- cv.glmnet(x_train, y_train, alpha = 0.5)


## Generate Predictions

In [6]:
# Predictions

baseline_pred <- predict(baseline_model, newdata = test_data)

refined_pred <- predict(refined_model, newdata = test_data)

ridge_pred <- predict(ridge_model, s = "lambda.min", newx = x_test)

lasso_pred <- predict(lasso_model, s = "lambda.min", newx = x_test)

elastic_pred <- predict(elastic_model, s = "lambda.min", newx = x_test)

## Evaluate Model Performance

We compare models using:

- RMSE
- R-squared

In [7]:
# RMSE function
rmse <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2))
}

# R2 function
r2 <- function(actual, predicted) {
  cor(actual, predicted)^2
}

# Compute metrics
results <- data.frame(
  Model = c("Baseline", "Refined", "Ridge", "Lasso", "ElasticNet"),
  RMSE = c(
    rmse(test_data$rented_bike_count, baseline_pred),
    rmse(test_data$rented_bike_count, refined_pred),
    rmse(test_data$rented_bike_count, ridge_pred),
    rmse(test_data$rented_bike_count, lasso_pred),
    rmse(test_data$rented_bike_count, elastic_pred)
  ),
  R2 = c(
    r2(test_data$rented_bike_count, baseline_pred),
    r2(test_data$rented_bike_count, refined_pred),
    r2(test_data$rented_bike_count, ridge_pred),
    r2(test_data$rented_bike_count, lasso_pred),
    r2(test_data$rented_bike_count, elastic_pred)
  )
)

# Display results
results

Model,RMSE,R2
<chr>,<dbl>,<dbl>
Baseline,491.4609,0.4121109
Refined,472.4720,0.4568859
Ridge,426.2237,0.5588644
Lasso,417.4816,0.5758793
ElasticNet,417.4841,0.5758784


## Select Best Model

The best model is selected based on:

- Lowest RMSE
- Highest R-squared

Regularized models often provide better generalization performance.

In [8]:
# Identify best model based on RMSE

best_model <- results[which.min(results$RMSE), ]

best_model

,Model,RMSE,R2
,<chr>,<dbl>,<dbl>
4,Lasso,417.4816,0.5758793


In [9]:
lasso_coefs <- coef(lasso_model, s = "lambda.min")
print(lasso_coefs)

22 x 1 sparse Matrix of class "dgCMatrix"
                        lambda.min
(Intercept)          -7.433563e+01
temperature           4.446745e+01
humidity              1.426879e+01
wind_speed            2.649702e+01
visibility            2.343525e-03
solar_radiation      -9.462025e+01
rainfall             -4.559427e+01
snowfall              7.480727e+00
temperature_sq       -1.112613e-01
humidity_sq          -1.654615e-01
temp_humidity        -2.952829e-01
seasonsSummer         3.848711e+01
seasonsAutumn         1.327832e+02
seasonsWinter        -1.915398e+02
holidayHoliday       -1.167777e+02
day_of_weekTuesday    2.780146e+01
day_of_weekWednesday  6.596195e+01
day_of_weekThursday   2.757526e+01
day_of_weekFriday     4.564311e+01
day_of_weekSaturday  -3.058536e+01
day_of_weekSunday    -9.602004e+01
hour                  2.795234e+01

## Interpretation

From model comparison:

- Refined model improves over baseline by capturing non-linear relationships
- Regularized models reduce overfitting
- Elastic Net often balances bias and variance effectively

The selected model provides the best trade-off between accuracy and generalization.

## Preparing for Deployment

The selected model can be used in the prediction pipeline and integrated into the Shiny dashboard.

This ensures that predictions are generated consistently using the best-performing model. 

Note that the Lab had provided a pre trained model which is used in the Shiny App.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta </span>  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on comparing multiple regression models and selecting the best-performing model for deployment.

The workflow follows <span style="color:blue">IBM Skills Network </span> instructional labs on regression model evaluation and selection.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo </span>  
- <span style="color:blue">Jeff Grossman</span>  

---

## Project Context

This notebook represents the final model selection stage in the end-to-end data science pipeline:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Baseline Model Development  
- Model Refinement  
- Model Evaluation & Diagnostics  
- Feature Importance Analysis  
- **Final Model Selection**  
- Deployment (R Shiny Dashboard)

---

## Notes

Selecting the optimal model ensures that the prediction system achieves high accuracy while maintaining robustness and generalization capability.

---